# The Internal Parking Lot

### Investment Banking — Banking-AI


            ## Step 0 — Notebook Header

            **Prerequisites:** Basic Python, familiarity with market-impact concepts, and comfort with regression-style decision support.

            **What You Will Learn:**

            - Describe how overlap, urgency, and transaction costs influence internal crossing decisions.
- Compare a simple overlap-based estimate against a richer regression model for internal-match savings.
- Construct a basic order-pairing output that shows when internal parking is preferable to external routing.

            **Estimated time:** 45 minutes

            **Collection context:** Investment-banking notebooks that connect prediction, optimization, and operational decision support in capital-markets workflows.


## Step 1 — Investment-Banking Problem Introduction

**Why this problem matters:** Routing every order to the market can create unnecessary impact, commissions, and taxes when a firm may already have a natural internal cross.

**Operational question:** Should the order be parked internally or routed out, and what savings might the internal match create?

**Primary stakeholders:** trading desks, portfolio managers, execution analysts, and transition-management teams

**Decision supported:** Estimate the value of internal matching so traders can decide whether patience is worth the potential savings.

**Comprehension check:** Before looking at the data, write one sentence describing why the best internal match is not always the largest one.

**Domain framing note:** This notebook is educational and synthetic. It demonstrates decision support for investment-banking and adjacent institutional workflows, not production approval or investment advice.


## Step 2 — Environment Setup

Use synthetic order-pair features and lightweight regression so the matching logic remains visible and easy to adapt.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
N_PAIRS = 1400

print("Environment ready for a synthetic internal-crossing workflow.")


## Step 3 — Data Creation & Context

We simulate candidate order pairs using overlap, spread costs, taxes, urgency, and signal decay to mimic the decision to hold or route externally.


In [ ]:
parking_df = pd.DataFrame({
    "internal_overlap_ratio": RNG.uniform(0.0, 1.0, N_PAIRS),
    "spread_cost_bps": RNG.uniform(2.0, 28.0, N_PAIRS),
    "tax_cost_bps": RNG.uniform(0.0, 16.0, N_PAIRS),
    "commission_bps": RNG.uniform(0.5, 8.0, N_PAIRS),
    "urgency_score": RNG.uniform(0.0, 1.0, N_PAIRS),
    "signal_decay_score": RNG.uniform(0.0, 1.0, N_PAIRS),
    "borrow_constraint_score": RNG.uniform(0.0, 1.0, N_PAIRS),
    "imbalance_penalty_score": RNG.uniform(0.0, 1.0, N_PAIRS),
})

savings_signal = (
    26 * parking_df["internal_overlap_ratio"]
    + 0.8 * parking_df["spread_cost_bps"]
    + 0.6 * parking_df["tax_cost_bps"]
    + 0.5 * parking_df["commission_bps"]
    - 10 * parking_df["urgency_score"]
    - 9 * parking_df["signal_decay_score"]
    - 7 * parking_df["borrow_constraint_score"]
    - 6 * parking_df["imbalance_penalty_score"]
    + RNG.normal(0, 2.5, N_PAIRS)
)
parking_df["internal_savings_bps"] = savings_signal.clip(-8.0, 45.0)
print(parking_df.head(3).round(3).to_string(index=False))


## Step 4 — Exploratory Data Analysis

Inspect how overlap and expected savings move together before the model adds nuance from urgency and market-friction signals.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.scatterplot(
    data=parking_df,
    x="internal_overlap_ratio",
    y="internal_savings_bps",
    alpha=0.7,
    ax=axes[0],
)
axes[0].set_title("Overlap Ratio vs. Internal Savings")
axes[0].set_xlabel("Internal Overlap Ratio")
axes[0].set_ylabel("Estimated Savings (bps)")

sns.histplot(data=parking_df, x="internal_savings_bps", bins=24, kde=True, ax=axes[1])
axes[1].set_title("Distribution of Internal Savings")
axes[1].set_xlabel("Estimated Savings (bps)")

plt.tight_layout()
plt.show()
plt.close()


Alt-Text: The EDA shows that higher overlap often improves expected internal-cross savings, but the spread in outcomes hints that urgency and other frictions still matter.


## Step 5 — Feature Engineering

We engineer a market-impact cost feature and a patience score to mirror the way execution teams reason about waiting versus crossing.


In [ ]:
analysis_df = parking_df.copy()
analysis_df["market_impact_cost_bps"] = (
    analysis_df["spread_cost_bps"]
    + analysis_df["tax_cost_bps"]
    + analysis_df["commission_bps"]
)
analysis_df["patience_score"] = 1 - (
    analysis_df["urgency_score"] + analysis_df["signal_decay_score"]
) / 2

feature_columns = [
    "internal_overlap_ratio",
    "spread_cost_bps",
    "tax_cost_bps",
    "commission_bps",
    "urgency_score",
    "signal_decay_score",
    "borrow_constraint_score",
    "imbalance_penalty_score",
    "market_impact_cost_bps",
    "patience_score",
]

print(analysis_df[feature_columns].head(3).round(3).to_string(index=False))


## Step 6 — Baseline Establishment

A simple desk baseline estimates savings from overlap alone, ignoring urgency and cost nuance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    analysis_df[feature_columns],
    analysis_df["internal_savings_bps"],
    test_size=0.25,
    random_state=RANDOM_SEED,
)

baseline_pred = 25 * X_test["internal_overlap_ratio"]
print(f"Baseline MAE: {mean_absolute_error(y_test, baseline_pred):.3f}")
print(f"Baseline RMSE: {np.sqrt(mean_squared_error(y_test, baseline_pred)):.3f}")


## Step 7 — Model Training & Evaluation

Train a richer regression model that accounts for patience, friction, and borrow constraints before recommending internal parking.


In [ ]:
parking_model = RandomForestRegressor(
    n_estimators=260,
    min_samples_leaf=4,
    random_state=RANDOM_SEED,
)
parking_model.fit(X_train, y_train)
model_pred = parking_model.predict(X_test)

print(f"Model MAE: {mean_absolute_error(y_test, model_pred):.3f}")
print(f"Model RMSE: {np.sqrt(mean_squared_error(y_test, model_pred)):.3f}")
print(f"Model R^2: {r2_score(y_test, model_pred):.3f}")


## Step 8 — Interpretability & Explainability

Feature importance reveals whether the model values overlap and market friction in the same balance an experienced trader would expect.


In [ ]:
importance_frame = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": parking_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

sns.barplot(data=importance_frame, y="feature", x="importance", color="#E07A5F")
plt.title("Drivers of Internal Parking Savings")
plt.xlabel("Model Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()
plt.close()

print(importance_frame.head(6).round(3).to_string(index=False))


## Step 9 — Operational Application

Use the prediction in a simple order-pairing table so the desk can decide which orders to cross internally and which to route out.


In [ ]:
order_pairs = pd.DataFrame(
    {
        "internal_overlap_ratio": [0.92, 0.34, 0.71],
        "spread_cost_bps": [12, 7, 16],
        "tax_cost_bps": [6, 2, 8],
        "commission_bps": [3, 2, 4],
        "urgency_score": [0.18, 0.84, 0.36],
        "signal_decay_score": [0.22, 0.76, 0.28],
        "borrow_constraint_score": [0.12, 0.18, 0.44],
        "imbalance_penalty_score": [0.14, 0.55, 0.26],
    },
    index=["patient_cross", "route_now", "balanced_case"],
)
order_pairs["market_impact_cost_bps"] = (
    order_pairs["spread_cost_bps"]
    + order_pairs["tax_cost_bps"]
    + order_pairs["commission_bps"]
)
order_pairs["patience_score"] = 1 - (
    order_pairs["urgency_score"] + order_pairs["signal_decay_score"]
) / 2
order_pairs["predicted_savings_bps"] = parking_model.predict(order_pairs[feature_columns])
order_pairs["routing_decision"] = np.where(
    order_pairs["predicted_savings_bps"] >= 8.0, "Park internally", "Route externally"
)

print(order_pairs.round(3).to_string())


            ## Step 10 — Summary & Reflection

            **What you accomplished:** You built a synthetic internal-crossing workflow that estimates when patience and internal matching are worth more than immediate external routing.

            **Limitations to note:**

            - The order-pair data is synthetic and does not capture real venue, latency, or portfolio-constraint complexity.
- The predicted savings are educational estimates rather than live transaction-cost forecasts.
- Real deployment would require policy checks, conflict management, and detailed execution governance.

            **Ethical reflection:** Internal matching can reduce costs, but firms still need strong controls to ensure fair treatment across portfolios and clients.

            **Reflection questions:**

            1. Which real execution constraint would most change the parking decision even if the expected savings looked attractive?
2. How would you extend the workflow from one order pair to a larger portfolio-crossing problem?
3. When should a desk override a high predicted savings estimate and route immediately anyway?


            ## References

            - Breiman, L. (2001). Random Forests.
- Scikit-learn user guide: regression metrics and tree-based models.
- Scenario inspiration: transition-management, crossing-network, and market-impact analysis workflows.
